In [3]:
import cv2
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython import display as ipd
from pathlib import Path

In [5]:
from src.utils.utils import ComputerVision
from src.data.loaders import DataLoader

In [6]:
ROOT_DIR = Path.cwd().parent.parent
# sys.path.insert(0, str(ROOT_DIR / "src"))

print(ROOT_DIR)

/home/samuel-linux/master-degree-project


In [7]:
OUTPUT_DIR = ROOT_DIR / "outputs" / "notebooks" / "20260610-01-frames-brightness-analysis"
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_DIR = ROOT_DIR / "datasets" / "RLVS" / "Violence"
INPUT_DIR.mkdir(exist_ok=True)

In [8]:
cv = ComputerVision()
dataset_loader = DataLoader(INPUT_DIR)

In [9]:
video_files = dataset_loader.get_video_files()
print(f"Found {len(video_files)} video files in {INPUT_DIR}")
print(f"Video files: {[video.name for video in video_files]}")

Found 40 video files in /home/samuel-linux/master-degree-project/datasets/RLVS/Violence
Video files: ['V_17.mp4', 'V_1.mp4', 'V_33.mp4', 'V_6.mp4', 'V_16.mp4', 'V_29.mp4', 'V_40.mp4', 'V_22.mp4', 'V_15.mp4', 'V_7.mp4', 'V_24.mp4', 'V_28.mp4', 'V_5.mp4', 'V_8.mp4', 'V_34.mp4', 'V_26.mp4', 'V_39.mp4', 'V_20.mp4', 'V_38.mp4', 'V_25.mp4', 'V_32.mp4', 'V_12.mp4', 'V_36.mp4', 'V_3.mp4', 'V_4.mp4', 'V_37.mp4', 'V_23.mp4', 'V_30.mp4', 'V_31.mp4', 'V_19.mp4', 'V_14.mp4', 'V_27.mp4', 'V_21.mp4', 'V_35.mp4', 'V_9.mp4', 'V_13.mp4', 'V_10.mp4', 'V_18.mp4', 'V_11.mp4', 'V_2.mp4']


In [10]:
def annotate_video(input_path, output_path, thresholds):
    cap = cv2.VideoCapture(input_path)
    
    if not cap.isOpened():
        print("Error: Could not open the video file.")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    frame_counts = {'Day': 0, 'Evening': 0, 'Night': 0}
    frame_index = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        brightness = cv.calculate_brightness(frame)
        classification = cv.classify_frame(brightness, thresholds)
        
        # print(f"Frame {frame_index}: Brightness={brightness:.2f}, Classification={classification}")
        
        frame_counts[classification] += 1
        text = f"Frame: {frame_index}, Time: {classification}"
        
        cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"FPS: {fps:.2f}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
        
        out.write(frame)
        frame_index += 1
        
    cap.release()
    out.release()
    total_frames = sum(frame_counts.values())
    percentages = {key: (count / total_frames) * 100 for key, count in frame_counts.items()}
    print(f"Annotated video saved to {output_path}")
    print(f"Day: {percentages['Day']:.2f}%, Evening: {percentages['Evening']:.2f}%, Night: {percentages['Night']:.2f}%")

In [11]:
thresholds = {
    'day': 120,
    'evening': 70,
    'night': 0
}

input_video_path = INPUT_DIR / "V_20.mp4"
output_video_path = OUTPUT_DIR / "annotated_video.avi"
annotate_video(input_video_path, output_video_path, thresholds)

Annotated video saved to /home/samuel-linux/master-degree-project/outputs/notebooks/20260610-01-frames-brightness-analysis/annotated_video.avi
Day: 16.33%, Evening: 83.67%, Night: 0.00%
